In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# UCI repository CSV link (hosted on archive.ics.uci.edu)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names from UCI documentation
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target"
]

df = pd.read_csv(url, names=columns)

# Replace missing values ("?") with NaN and drop rows with NaN
df = df.replace("?", pd.NA)
df = df.dropna()
df = df.astype(float)

# Convert target to binary: 0 = no disease, 1 = disease present
df["target"] = (df["target"] > 0).astype(int)
data = df
X = data.drop("target",axis = 1)
y = data["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

param_grid = [
    {
        'max_iter': [5000]
    }
]

grid = GridSearchCV(LogisticRegression(),param_grid,scoring = 'accuracy',cv = 5,n_jobs = -1)
grid.fit(X_train,y_train)
model = grid.best_estimator_

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print(f"Accuracy: {accuracy}")
conf_matrix = confusion_matrix(y_test,y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix = conf_matrix,display_labels = model.classes_)
disp.plot()


OSError: [WinError 1455] The paging file is too small for this operation to complete

In [ ]:
import numpy as np
probs = model.predict_proba(X_test)[:,1]
min_fn = len(y_test)
min_t = 1
for t in np.linspace(0,1,100):
    y_pred = (probs >= t).astype(int)
    tp, fn, fp, tn = confusion_matrix(y_test,y_pred).ravel()
    if fn < min_fn:
        min_t = t
        min_fn = fn
        conf = confusion_matrix(y_test,y_pred)
ConfusionMatrixDisplay(confusion_matrix = conf,display_labels = model.classes_).plot()
